In [1]:
import requests
import psycopg2
import time
from datetime import datetime

In [2]:
# PostgreSQL 연결
conn = psycopg2.connect(
    host="localhost",
    dbname="postgres",
    user="postgres",
    password="1111",
    port=5432
)
cur = conn.cursor()

In [3]:
# 테이블 생성 (거래량 컬럼 추가)
cur.execute("""
CREATE TABLE IF NOT EXISTS pi_usdt_price_log (
    id SERIAL PRIMARY KEY,
    price_usd NUMERIC(12,6),
    usd_krw NUMERIC(12,3),
    price_krw NUMERIC(12,3),
    volume_usd NUMERIC(18,3),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
conn.commit()


def get_pi_data():
    """Pi 코인의 USD 가격과 24시간 거래량을 CoinGecko에서 가져옴"""
    url = "https://api.coingecko.com/api/v3/coins/pi-network"
    res = requests.get(url, timeout=5)
    data = res.json()

    price_usd = float(data["market_data"]["current_price"]["usd"])
    volume_usd = float(data["market_data"]["total_volume"]["usd"])

    return price_usd, volume_usd


def get_usd_to_krw():
    """USD→KRW 환율을 가져옴 (백업 API 포함)"""
    try:
        url = "https://api.exchangerate.host/latest?base=USD&symbols=KRW"
        res = requests.get(url, timeout=5)
        data = res.json()
        if "rates" in data and "KRW" in data["rates"]:
            return float(data["rates"]["KRW"])
        else:
            raise ValueError("rates 키 없음")
    except Exception:
        # 백업 API
        backup = requests.get("https://open.er-api.com/v6/latest/USD", timeout=5).json()
        return float(backup["rates"]["KRW"])

In [ ]:
# 1분마다 반복 실행
while True:
    try:
        price_usd, volume_usd = get_pi_data()
        usd_krw = get_usd_to_krw()
        price_krw = price_usd * usd_krw
        now = datetime.now()

        cur.execute("""
            INSERT INTO pi_usdt_price_log 
                (price_usd, usd_krw, price_krw, volume_usd, created_at)
            VALUES (%s, %s, %s, %s, %s)
        """, (price_usd, usd_krw, price_krw, volume_usd, now))
        conn.commit()

        print(f"[{now}] ✅ Pi: ${price_usd:.6f} | ₩{price_krw:.2f} | 거래량: ${volume_usd:,.2f} | 환율: {usd_krw:.2f}")

    except Exception as e:
        print("❌ 오류 발생:", e)
        conn.rollback()

    time.sleep(60)

[2025-10-15 07:59:23.549757] ✅ Pi: $0.215971 | ₩308.03 | 거래량: $51,687,037.00 | 환율: 1426.24
[2025-10-15 08:00:27.738329] ✅ Pi: $0.215929 | ₩307.97 | 거래량: $51,670,319.00 | 환율: 1426.24
[2025-10-15 08:01:33.018838] ✅ Pi: $0.215881 | ₩307.90 | 거래량: $51,666,491.00 | 환율: 1426.24
[2025-10-15 08:02:36.542124] ✅ Pi: $0.215881 | ₩307.90 | 거래량: $51,666,491.00 | 환율: 1426.24
[2025-10-15 08:03:40.979490] ✅ Pi: $0.216139 | ₩308.27 | 거래량: $51,679,337.00 | 환율: 1426.24
[2025-10-15 08:04:44.199785] ✅ Pi: $0.216094 | ₩308.20 | 거래량: $51,641,203.00 | 환율: 1426.24
[2025-10-15 08:05:48.808095] ✅ Pi: $0.216230 | ₩308.40 | 거래량: $51,650,007.00 | 환율: 1426.24
[2025-10-15 08:06:51.675372] ✅ Pi: $0.216230 | ₩308.40 | 거래량: $51,650,007.00 | 환율: 1426.24
[2025-10-15 08:07:56.140944] ✅ Pi: $0.216356 | ₩308.58 | 거래량: $51,638,817.00 | 환율: 1426.24
[2025-10-15 08:08:59.360172] ✅ Pi: $0.216356 | ₩308.58 | 거래량: $51,638,817.00 | 환율: 1426.24
[2025-10-15 08:10:06.605280] ✅ Pi: $0.216527 | ₩308.82 | 거래량: $51,628,292.00 | 환율: 1426.24